# Step 1 - Ingest Competition CSVs

### Inputs
`data/0_raw/competition/exp{58,72,73}_{wide,long}.csv` (copies of the reference exports from `docs/codebooks/Reference/`).

### Outputs
`data/0_raw/competition/wide_all.csv`, `data/0_raw/competition/long_all.csv`, the three studies stacked with a `study` and `n_ranks` column.

This step only assembles the raw exports into one union-schema table per shape (wide, long). No cleaning happens here; all filtering lives in Step 2 (`clean_competition`).

### Notes
- Schemas differ across studies. exp72 and exp73 are 6-person contests (ranks 2-5 elicited); exp58 is a 10-person contest (ranks 2, 3, 8, 9 elicited; rank 5 is also elicited per the authors' data.xlsx).
- We stack with `pd.concat` so columns missing from a given study become `NaN` in the union schema.
- `rlabel`: `w` = word format, `n` = numerical (wording factor).
- `ordercon`: 1 = ascending, 2 = descending (order factor).

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "run_all.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not find project root (run_all.py)")
    ROOT = ROOT.parent

RAW = ROOT / "data" / "0_raw" / "competition"
OUT_LOG = ROOT / "output" / "0_raw" / "ingest_competition_csvs"
OUT_LOG.mkdir(parents=True, exist_ok=True)

STUDIES = {
    "exp72": {"n_ranks": 6, "elicited": [2, 3, 4, 5]},
    "exp73": {"n_ranks": 6, "elicited": [2, 3, 4, 5]},
    "exp58": {"n_ranks": 10, "elicited": [2, 3, 8, 9]},
}
print(f"Project root: {ROOT}")
print(f"Raw dir:      {RAW}")

Project root: C:\Users\clint\Desktop\Rank_Pref_Replication
Raw dir:      C:\Users\clint\Desktop\Rank_Pref_Replication\data\0_raw\competition


## Wide files, one row per participant

In [2]:
# Read each study's wide CSV, tag it with `study` and `n_ranks`, then stack into
# one union-schema frame. `utf-8-sig` strips the byte-order mark some Excel exports prepend.
wide_frames = []
for study, meta in STUDIES.items():
    df = pd.read_csv(RAW / f"{study}_wide.csv", encoding="utf-8-sig")
    df.insert(0, "study", study)
    df.insert(1, "n_ranks", meta["n_ranks"])
    print(f"{study}: {df.shape[0]:>4} rows, {df.shape[1]:>2} cols")
    wide_frames.append(df)

wide_all = pd.concat(wide_frames, ignore_index=True, sort=False)
print(f"\nstacked wide_all: {wide_all.shape}")
print(f"union of columns: {list(wide_all.columns)}")

exp72: 2088 rows, 31 cols
exp73: 1065 rows, 31 cols
exp58:  446 rows, 29 cols

stacked wide_all: (3599, 38)
union of columns: ['study', 'n_ranks', 'row_id', 'pid', 'rlabel', 'ordercon', 'domain', 'ip', 'dateadded', 'datemodified', 'numsecs', 'nummins', 'code', 'rankOrder', 'rank2indif', 'rank3indif', 'rank4indif', 'rank5indif', 'actuallyrank', 'feel1', 'feel2', 'feel5', 'feel6', 'feeldrop', 'feelrise', 'howcompetitive', 'howfitm', 'howfitp', 'gender', 'age', 'ethnicity', 'rank6indif', 'rank7indif', 'rank8indif', 'rank9indif', 'feelfirst', 'feellast', 'howfit']


In [3]:
# Sanity check on key analysis columns
for col in ["rank2indif", "rank3indif", "rank4indif", "rank5indif", "rank8indif", "rank9indif"]:
    if col in wide_all.columns:
        miss = wide_all.groupby("study")[col].apply(lambda s: s.isna().sum())
        print(f"{col} missing by study:\n{miss}\n")

rank2indif missing by study:
study
exp58    15
exp72    60
exp73    49
Name: rank2indif, dtype: int64

rank3indif missing by study:
study
exp58    15
exp72    67
exp73    51
Name: rank3indif, dtype: int64

rank4indif missing by study:
study
exp58    446
exp72     68
exp73     50
Name: rank4indif, dtype: int64

rank5indif missing by study:
study
exp58    13
exp72    62
exp73    45
Name: rank5indif, dtype: int64

rank8indif missing by study:
study
exp58      15
exp72    2088
exp73    1065
Name: rank8indif, dtype: int64

rank9indif missing by study:
study
exp58      15
exp72    2088
exp73    1065
Name: rank9indif, dtype: int64



## Long files, one row per bisection trial

In [4]:
# Same stacking for the long (per-trial) files. These hold the raw bisection
# trials behind each elicited indifference probability; not used by the wide
# cleaning, kept for completeness.
long_frames = []
for study in STUDIES:
    df = pd.read_csv(RAW / f"{study}_long.csv", encoding="utf-8-sig")
    df.insert(0, "study", study)
    print(f"{study}: {df.shape[0]:>5} rows, {df.shape[1]:>2} cols, ranks={sorted(df['ranknum'].unique())}, qnums={sorted(df['qnum'].unique())}")
    long_frames.append(df)

long_all = pd.concat(long_frames, ignore_index=True, sort=False)
print(f"\nstacked long_all: {long_all.shape}")

exp72: 40783 rows, 13 cols, ranks=[np.int64(2), np.int64(3), np.int64(4), np.int64(5)], qnums=[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
exp73: 20444 rows, 13 cols, ranks=[np.int64(2), np.int64(3), np.int64(4), np.int64(5)], qnums=[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
exp58: 10905 rows, 13 cols, ranks=[np.int64(2), np.int64(3), np.int64(5), np.int64(8), np.int64(9)], qnums=[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

stacked long_all: (72132, 13)


In [5]:
# Each (pid, ranknum) should have exactly 5 bisection trials (qnum 1..5).
trials = long_all.groupby(["study", "pid", "ranknum"]).size().rename("n_trials")
print("Distribution of trials per (pid, ranknum):")
print(trials.value_counts().sort_index())
off = trials[trials != 5]
if len(off):
    print(f"\n{len(off)} (pid, rank) groups do not have 5 trials (preview):")
    print(off.head())

Distribution of trials per (pid, ranknum):
n_trials
1         9
2         7
3         8
4        11
5     14255
6        20
7         4
8         7
9         6
10       47
11        1
12        1
15        1
Name: count, dtype: int64

122 (pid, rank) groups do not have 5 trials (preview):
study  pid    ranknum
exp58  20037  2           9
       20076  2           8
       20082  9           1
       20183  2          10
              3          10
Name: n_trials, dtype: int64


## Write stacked outputs

In [6]:
wide_out = RAW / "wide_all.csv"
long_out = RAW / "long_all.csv"
wide_all.to_csv(wide_out, index=False)
long_all.to_csv(long_out, index=False)
print(f"wrote {wide_out.relative_to(ROOT)}  ({wide_all.shape[0]} rows)")
print(f"wrote {long_out.relative_to(ROOT)}  ({long_all.shape[0]} rows)")

# Brief log
log = OUT_LOG / "ingest_summary.txt"
with open(log, "w") as f:
    f.write(f"wide_all: {wide_all.shape}\n")
    f.write(f"long_all: {long_all.shape}\n")
    for study in STUDIES:
        n = (wide_all['study'] == study).sum()
        f.write(f"  {study}: {n} participants\n")
print(f"wrote {log.relative_to(ROOT)}")

wrote data\0_raw\competition\wide_all.csv  (3599 rows)
wrote data\0_raw\competition\long_all.csv  (72132 rows)
wrote output\0_raw\ingest_competition_csvs\ingest_summary.txt
